
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Appendix - Python Auto Loader
### Extra material, not part of a live teach.

In this demonstration we will introduce running Auto Loader in Python for incremental ingestion. In this example you will be execute Auto Loader manually to incrementally ingest the data.


## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.3/images/20261001T131157Z/Data Ingestion with LakeFlow Connect/Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>



## A. Classroom Setup

Run the following cell to configure your working environment for this notebook.

In [0]:
%run ../Includes/Classroom-Setup-Auto-Loader

View the available file(s) in your `/Volumes/labuser123_678/data_ingestion/csv_files_autoloader_source` volume. Notice only one file exists.

In [0]:
%python
spark.sql(f'LIST "/Volumes/{my_catalog}/data_ingestion/csv_files_autoloader_source"').display()


[What is Auto Loader?](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/)

[Using Auto Loader with Unity Catalog](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/unity-catalog)

[Auto Loader options](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/options#csv-options)

Below is an example of Auto Loader.

In [0]:
%python

## Create a volume to store the Auto Loader checkpoint files
spark.sql(f'CREATE VOLUME IF NOT EXISTS {my_catalog}.data_ingestion.auto_loader_files')

## Set checkpoint location to the volume from above
checkpoint_file_location = f'/Volumes/{my_catalog}/data_ingestion/auto_loader_files'

## Incrementally (or stream) data using Auto Loader
(spark
 .readStream
   .format("cloudFiles")
   .option("cloudFiles.format", "csv")
   .option("header", "true")
   .option("sep", "|")
   .option("inferSchema", "true")
   .option("cloudFiles.schemaLocation", f"{checkpoint_file_location}")
   .load(f"/Volumes/{my_catalog}/data_ingestion/csv_files_autoloader_source/")
 .writeStream
   .option("checkpointLocation", f"{checkpoint_file_location}")
   .trigger(once=True)
   .toTable(f"{my_catalog}.data_ingestion.python_csv_autoloader")
)

View the new table **python_csv_autoloader**. Notice the data was ingested into a table and contains 3,149 rows.

In [0]:
SELECT *
FROM python_csv_autoloader;

Add a CSV file to your `/Volumes/dbacademy/labuser/csv_files_autoloader_source/` volume.

In [0]:
%python
copy_files(copy_from = '/Volumes/dbacademy_ecommerce/v01/raw/sales-csv/', copy_to = f"/Volumes/{my_catalog}/data_ingestion/csv_files_autoloader_source/", n=2)

Confirm your volume contains 2 CSV files.

In [0]:
%python
spark.sql(f'LIST "/Volumes/{my_catalog}/data_ingestion/csv_files_autoloader_source"').display()

Rerun your Auto Loader ingestion from above (pasted for you below) to incrementally ingest only the new file.

In [0]:
%python

checkpoint_file_location = f'/Volumes/{my_catalog}/data_ingestion/auto_loader_files'

(spark
 .readStream
   .format("cloudFiles")
   .option("cloudFiles.format", "csv")
   .option("header", "true")
   .option("sep", "|")
   .option("inferSchema", "true")
   .option("cloudFiles.schemaLocation", f"{checkpoint_file_location}")
   .load(f"/Volumes/{my_catalog}/data_ingestion/csv_files_autoloader_source/")
 .writeStream
   .option("checkpointLocation", f"{checkpoint_file_location}")
   .trigger(once=True)
   .toTable(f"{my_catalog}.data_ingestion.python_csv_autoloader")
)

View the **python_csv_autoloader** table. Notice that it now contains 6,081 rows.

In [0]:
SELECT * 
FROM python_csv_autoloader;

View the history of the **python_csv_autoloader** table. Notice the two **STREAMING UPDATES**. In the **operationMetrics** column you can see how many rows were ingestion in each streaming update. Notice that it only is ingestion new files.

In [0]:
DESCRIBE HISTORY python_csv_autoloader;

Drop the **python_csv_autoloader** table.

In [0]:
DROP TABLE IF EXISTS python_csv_autoloader;

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>